In [1]:
%pip -q install selenium webdriver-manager beautifulsoup4

# Google Colab用。ローカルでChromeが導入済みの場合は実行不要です。
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /tmp/google-chrome.deb
!apt-get update -qq
!apt-get install -y -qq /tmp/google-chrome.deb
"""日経平均株価をグラフから取得する（課題004）。"""
import re
from datetime import datetime
from time import perf_counter

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

URL = "https://www.nikkei.com/smartchart/?code=N101/T&timeframe=6m&interval=1Day&eventsShow=1"
LEGEND = ".modcharts-legend.first"
CHART = 'canvas.modcharts-panel-root[aria-label="Chart for NIKKEI 225 INDEX (NIK)"]'


def extract_stock_data(html):
    """HTMLから[日付, 始値, 高値, 安値, 終値]を返す。未表示ならNone。"""
    soup = BeautifulSoup(html, "html.parser")
    legend = soup.select_one(LEGEND)
    if legend is None:
        return None
    date_element = legend.select_one(".chartworks-legend-timestamp")
    if date_element is None:
        return None
    match = re.search(r"(\d{4})年\s*(\d{1,2})月\s*(\d{1,2})日", date_element.get_text())
    if match is None:
        return None
    date = datetime(*map(int, match.groups())).strftime("%Y-%m-%d")
    values = []
    for field in ("o", "h", "l", "c"):
        element = legend.select_one(f".chartworks-legend-ohlc-{field}")
        if element is None:
            return None
        value = element.get_text(strip=True).replace(",", "")
        if not re.fullmatch(r"\d+(?:\.\d+)?", value):
            return None
        values.append(float(value))
    return [date, *values]


def get_stock_values(driver, url):
    """グラフの右端から左端へ1pxずつ移動し、日付の重複を除いて返す。"""
    driver.get(url)
    wait = WebDriverWait(driver, 60)
    chart = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, CHART)))
    wait.until(lambda d: extract_stock_data(d.page_source))

    # Seleniumのオフセット原点は要素の中央。右境界より1px内側へ移動する。
    width = int(chart.rect["width"])
    right_offset = width // 2 - 1
    ActionChains(driver).move_to_element(chart).move_by_offset(right_offset, 0).pause(0.1).perform()
    rows = {}
    # 画面上のHTMLを毎回解析する。右側の価格軸などは同じ日付になるので除外する。
    for pixel in range(width - 1):
        if pixel:
            ActionChains(driver, duration=20).move_by_offset(-1, 0).pause(0.03).perform()
        row = extract_stock_data(driver.page_source)
        if row is not None:
            rows[row[0]] = row
    if not rows:
        raise RuntimeError("株価を取得できませんでした。ページの表示やHTML構造を確認してください。")
    return sorted(rows.values(), key=lambda row: row[0])


def main():
    started_at = datetime.now()
    started = perf_counter()
    print("開始時間:", started_at.strftime("%Y-%m-%d %H:%M:%S"))
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1440,1000")
    options.add_argument("--lang=ja-JP")
    driver = None
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        rows = get_stock_values(driver, URL)
        print("日付, 始値, 高値, 安値, 終値")
        for row in rows:
            print(row[0], *(f"{value:.2f}" for value in row[1:]), sep=", ")
        print(f"取得件数: {len(rows)}件")
    finally:
        if driver is not None:
            driver.quit()
        print("終了時間:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
        print(f"所要時間: {perf_counter() - started:.2f}秒")


if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 3.0 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package at-spi2-common.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../0-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatk1.0-0t64:amd64.
Preparing to unpack .../1-libatk1.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatk1.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../2-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpackin